# 품질 AI + AF 연결 비교

팀 BUT RF/CNN을 재현하여 각각 기존 AF 모델 앞에 연결합니다. 가중치가 Git에 없어서 재학습하며 기존 AF 모델은 고정합니다. 판정 보류는 비AF로 계산하지 않습니다. MIMIC·6종부터 평가하며 Longterm은 이번 학습/평가에 포함하지 않습니다. RF는 Colab CPU, CNN·AF는 GPU를 사용합니다. 기존 시험군 재사용 개발 평가입니다.


In [ ]:
from pathlib import Path
import subprocess,sys,torch
assert torch.cuda.is_available(),'T4 GPU 필요'
from google.colab import drive
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
assert (base/'models/main_two_sources_v1/candidate/model.pt').is_file()
repo=Path('/content/capdyssey_quality_link')
if not repo.exists():subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
subprocess.run(['git','checkout','origin/main'],cwd=repo,check=True)
print('CODE',subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','wfdb','scikit-learn','joblib','pandas','matplotlib'],check=True)
def run_script(name,extra=[]):
    with (base/'quality_af_link.log').open('a') as log:
        with subprocess.Popen([sys.executable,'-u',str(repo/'scripts'/name),'--base',str(base),*extra],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
            for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            assert p.wait()==0,'실행 오류: 이후 단계 중단'


In [ ]:
run_script('train_team_quality.py')


In [ ]:
models=base/'models/team_quality_reproduction_v1'
out=base/'models/quality_af_fixed_v1'
run_script('evaluate_quality_af.py',['--rf',str(models/'rf/butppg_quality_rf.joblib'),'--cnn-dir',str(models/'cnn'),'--out',str(out)])


In [ ]:
import pandas as pd
display(pd.read_csv(out/'comparison.csv'))
print('통과 구간 성능과 전체 AF 검출률·보류율을 함께 판단하세요. 자동 최종본 승격 없음.')
